# Exploratory Data Analysis & Signal Analysis

This notebook explores the predictive structure of the BAF dataset before model development. The focus is on fraud prevalence, feature distributions, categorical risk differences, temporal drift, and early signal assessment.

In [3]:
#Data Loading

from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path(
    r"D:\bank-account-fraud-decision-system"
)

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"

df = pd.read_csv(
    RAW_DATA_DIR / "Base.csv"
)


# Data Cleaning 

df_clean = df.drop(
    columns=["device_fraud_count"]
).copy()

sentinel_features = [
    "prev_address_months_count",
    "current_address_months_count",
    "bank_months_count",
    "session_length_in_minutes",
    "device_distinct_emails_8w"
]

for col in sentinel_features:
    df_clean[f"{col}_missing"] = (
        df_clean[col] == -1
    ).astype(int)

    df_clean.loc[
        df_clean[col] == -1,
        col
    ] = np.nan

print(df_clean.shape)

(1000000, 36)


## 1. Fraud Prevalence

The target is highly imbalanced, so fraud-rate analysis is more informative than raw counts alone.

In [4]:
fraud_summary = (
    df_clean["fraud_bool"]
    .value_counts()
    .to_frame("count")
)

fraud_summary["percentage"] = (
    fraud_summary["count"]
    / len(df_clean)
    * 100
)

display(fraud_summary)

,count,percentage
fraud_bool,,
0,988971,98.8971
1,11029,1.1029


## 2. Numeric Feature Signal

Numeric features are screened using their correlation with the fraud target. Correlation is used only as an initial linear signal check and not as a complete measure of predictive usefulness.

In [7]:
numeric_cols = (
    df_clean
    .select_dtypes(include=["number"])
    .columns
    .tolist()
)

numeric_cols.remove("fraud_bool")

correlations = (
    df_clean[numeric_cols + ["fraud_bool"]]
    .corr(numeric_only=True)["fraud_bool"]
    .drop("fraud_bool")
    .sort_values(
        key=abs,
        ascending=False
    )
)

display(correlations.head(20))

credit_risk_score                    0.070624
proposed_credit_limit                0.068907
customer_age                         0.062959
keep_alive_session                  -0.050296
prev_address_months_count_missing    0.048070
income                               0.045079
date_of_birth_distinct_emails_4w    -0.043224
name_email_similarity               -0.036720
device_distinct_emails_8w            0.036542
has_other_cards                     -0.035156
phone_home_valid                    -0.035128
current_address_months_count         0.033480
bank_months_count_missing            0.029548
email_is_free                        0.027758
intended_balcon_amount              -0.024524
bank_months_count                    0.020929
prev_address_months_count            0.020083
velocity_6h                         -0.016892
foreign_request                      0.016885
month                                0.013250
Name: fraud_bool, dtype: float64

## 3. Categorical Fraud Rates

Categorical variables are evaluated by comparing fraud rates across their levels. This helps identify segments with elevated or reduced fraud risk.

In [8]:
categorical_cols = df_clean.select_dtypes(
    include=["object", "string"]
).columns.tolist()

for col in categorical_cols:
    print(f"\n{col}")

    result = (
        df_clean
        .groupby(col)["fraud_bool"]
        .agg(
            fraud_rate="mean",
            applications="count"
        )
        .sort_values(
            "fraud_rate",
            ascending=False
        )
    )

    result["fraud_rate_pct"] = (
        result["fraud_rate"] * 100
    ).round(4)

    display(result)


payment_type


,fraud_rate,applications,fraud_rate_pct
payment_type,,,
AC,0.016698,252071,1.6698
AB,0.011251,370554,1.1251
AD,0.010822,118837,1.0822
AA,0.005282,258249,0.5282
AE,0.003460,289,0.3460



employment_status


,fraud_rate,applications,fraud_rate_pct
employment_status,,,
CC,0.024684,37758,2.4684
CG,0.015453,453,1.5453
CA,0.012186,730252,1.2186
CB,0.006891,138288,0.6891
CD,0.003770,26522,0.3770
CE,0.002336,22693,0.2336
CF,0.001930,44034,0.1930



housing_status


,fraud_rate,applications,fraud_rate_pct
housing_status,,,
BA,0.037466,169675,3.7466
BD,0.008639,26161,0.8639
BC,0.006148,372143,0.6148
BB,0.006008,260965,0.6008
BF,0.004194,1669,0.4194
BG,0.003968,252,0.3968
BE,0.003441,169135,0.3441



source


,fraud_rate,applications,fraud_rate_pct
source,,,
TELEAPP,0.015891,7048,1.5891
INTERNET,0.010994,992952,1.0994



device_os


,fraud_rate,applications,fraud_rate_pct
device_os,,,
windows,0.024694,263506,2.4694
macintosh,0.013971,53826,1.3971
x11,0.011206,7228,1.1206
other,0.005760,342728,0.5760
linux,0.005155,332712,0.5155


## 4. Fraud vs Non-Fraud Feature Means

Selected numeric features are compared directly between fraudulent and non-fraudulent applications to identify practical differences in customer, transaction, and application behavior.

In [9]:
important_features = [
    "income",
    "name_email_similarity",
    "customer_age",
    "days_since_request",
    "intended_balcon_amount",
    "zip_count_4w",
    "velocity_6h",
    "velocity_24h",
    "velocity_4w",
    "bank_branch_count_8w",
    "credit_risk_score",
    "proposed_credit_limit",
    "session_length_in_minutes"
]

comparison = (
    df_clean
    .groupby("fraud_bool")[important_features]
    .mean()
    .T
)

comparison.columns = [
    "Non_Fraud_Mean",
    "Fraud_Mean"
]

comparison["Difference"] = (
    comparison["Fraud_Mean"]
    - comparison["Non_Fraud_Mean"]
)

display(comparison)

,Non_Fraud_Mean,Fraud_Mean,Difference
income,0.561313,0.686635,0.125322
name_email_similarity,0.494815,0.393161,-0.101654
customer_age,33.609125,40.858645,7.249520
days_since_request,1.025383,1.054615,0.029233
intended_balcon_amount,8.713907,3.962009,-4.751898
zip_count_4w,1572.138693,1622.311542,50.172850
velocity_6h,5670.664988,5183.913444,-486.751543
velocity_24h,4771.528849,4613.138798,-158.390051
velocity_4w,4857.444566,4755.844185,-101.600382
bank_branch_count_8w,184.923747,133.976426,-50.947321


## 5. Temporal Fraud Drift

Fraud prevalence is examined by month to understand how the target distribution changes over time. This is important for both model evaluation and deployment stability.

In [10]:
monthly_fraud = (
    df_clean
    .groupby("month")["fraud_bool"]
    .agg(
        applications="count",
        fraud_cases="sum",
        fraud_rate="mean"
    )
)

monthly_fraud["fraud_rate_pct"] = (
    monthly_fraud["fraud_rate"]
    * 100
).round(4)

display(monthly_fraud)

,applications,fraud_cases,fraud_rate,fraud_rate_pct
month,,,,
0,132440,1500,0.011326,1.1326
1,127620,1198,0.009387,0.9387
2,136979,1198,0.008746,0.8746
3,150936,1392,0.009222,0.9222
4,127691,1452,0.011371,1.1371
5,119323,1411,0.011825,1.1825
6,108168,1450,0.013405,1.3405
7,96843,1428,0.014746,1.4746


## 6. Missingness as Fraud Signal

Missing values may themselves contain predictive information. Fraud rates are compared between applications with observed and missing values for sentinel-coded features.

In [13]:
missing_flags = [
    "prev_address_months_count_missing",
    "current_address_months_count_missing",
    "bank_months_count_missing",
    "session_length_in_minutes_missing",
    "device_distinct_emails_8w_missing"
]

for col in missing_flags:

    result = (
        df_clean
        .groupby(col)["fraud_bool"]
        .agg(
            applications="count",
            fraud_cases="sum",
            fraud_rate="mean"
        )
    )

    result["fraud_rate_pct"] = (
        result["fraud_rate"] * 100
    ).round(4)

    print(f"\n{col}")
    display(result)


prev_address_months_count_missing


,applications,fraud_cases,fraud_rate,fraud_rate_pct
prev_address_months_count_missing,,,,
0,287080,895,0.003118,0.3118
1,712920,10134,0.014215,1.4215



current_address_months_count_missing


,applications,fraud_cases,fraud_rate,fraud_rate_pct
current_address_months_count_missing,,,,
0,995746,11015,0.011062,1.1062
1,4254,14,0.003291,0.3291



bank_months_count_missing


,applications,fraud_cases,fraud_rate,fraud_rate_pct
bank_months_count_missing,,,,
0,746365,6889,0.009230,0.9230
1,253635,4140,0.016323,1.6323



session_length_in_minutes_missing


,applications,fraud_cases,fraud_rate,fraud_rate_pct
session_length_in_minutes_missing,,,,
0,997985,11011,0.011033,1.1033
1,2015,18,0.008933,0.8933



device_distinct_emails_8w_missing


,applications,fraud_cases,fraud_rate,fraud_rate_pct
device_distinct_emails_8w_missing,,,,
0,999641,11025,0.011029,1.1029
1,359,4,0.011142,1.1142


## 7. Nonlinear Signal Screening

Pearson correlation captures mainly linear relationships. Mutual Information is therefore used as an additional screening method to identify potentially useful nonlinear relationships between numeric variables and fraud.

In [20]:
from sklearn.feature_selection import mutual_info_classif
from sklearn.impute import SimpleImputer

mi_sample = df_clean.sample(
    n=200_000,
    random_state=42
).copy()

In [21]:
mi_numeric_cols = (
    mi_sample
    .select_dtypes(include=["number"])
    .columns
    .tolist()
)

mi_numeric_cols.remove("fraud_bool")

X_mi = mi_sample[mi_numeric_cols]
y_mi = mi_sample["fraud_bool"]


mi_imputer = SimpleImputer(
    strategy="median"
)

X_mi_imputed = mi_imputer.fit_transform(
    X_mi
)

In [22]:
mi_scores = mutual_info_classif(
    X_mi_imputed,
    y_mi,
    random_state=42
)

mi_results = (
    pd.DataFrame({
        "feature": mi_numeric_cols,
        "mutual_information": mi_scores
    })
    .sort_values(
        "mutual_information",
        ascending=False
    )
)

display(
    mi_results.head(20)
)

,feature,mutual_information
25,prev_address_months_count_missing,0.043220
23,device_distinct_emails_8w,0.043088
22,keep_alive_session,0.038173
16,phone_mobile_valid,0.035642
14,email_is_free,0.034311
15,phone_home_valid,0.023488
2,prev_address_months_count,0.015438
19,proposed_credit_limit,0.012253
4,customer_age,0.011723
27,bank_months_count_missing,0.008453


## 8. EDA & Signal Analysis Summary

The exploratory analysis confirms that the BAF dataset contains meaningful predictive signal for fraud detection.

Several customer, application, device, and risk-related variables show measurable differences between fraudulent and non-fraudulent applications. Fraud rates also vary substantially across categorical segments such as housing status, employment status, payment type, and device operating system.

Missingness itself is informative for several features. In particular, missing previous-address history and missing bank-tenure information are associated with materially higher fraud rates, supporting the decision to retain explicit missingness indicators.

The mutual-information screening also identifies nonlinear relationships between fraud and several behavioral and device-related features, including address-history missingness, device email activity, session persistence, phone validity, and free-email usage.

Fraud prevalence increases over later monthly periods, reinforcing the need for chronological model evaluation rather than random splitting.

Overall, the dataset demonstrates sufficient predictive structure to proceed to feature engineering and model development.